In [1]:
import os
import json
import numpy as np
import re
from scipy.stats import t, norm

In [2]:
# Función que calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio
def calcular_kpis_promedio_con_ic_en_formato_json(directorio, nivel_confianza=95, guardar_en=None):
    """
    Calcula el promedio ± IC de todos los KPIs numéricos en archivos JSON de un directorio.
    Usa t-Student si n < 100, o normal estándar (z) si n >= 100.

    Args:
        directorio (str): Ruta a carpeta con archivos JSON.
        nivel_confianza (int or float): Nivel de confianza deseado (ej. 90, 95, 99).
        guardar_en (str or None): Ruta para guardar el resultado en un JSON (opcional).

    Returns:
        dict: Diccionario anidado con 'media ± error' y metadata de cálculo.
    """
    if not (50 <= nivel_confianza < 100):
        raise ValueError("El nivel de confianza debe estar entre 50 y 99.9")

    prob = nivel_confianza / 100
    json_files = [os.path.join(directorio, f) for f in os.listdir(directorio) if f.endswith(".json")]
    n = len(json_files)
    if n == 0:
        raise ValueError("No se encontraron archivos JSON en el directorio.")

    # Usar el primer archivo como referencia de estructura
    with open(json_files[0], "r") as f:
        ejemplo = json.load(f)

    def extraer_rutas(d, prefijo=""):
        rutas = []
        for k, v in d.items():
            ruta = f"{prefijo}.{k}" if prefijo else k
            if isinstance(v, dict):
                rutas += extraer_rutas(v, ruta)
            elif isinstance(v, (int, float)):
                rutas.append(ruta)
        return rutas

    kpis_ruta = extraer_rutas(ejemplo)

    # Recolectar valores de cada KPI
    data = {kpi: [] for kpi in kpis_ruta}
    for file in json_files:
        with open(file, "r") as f:
            contenido = json.load(f)
            for kpi in kpis_ruta:
                try:
                    val = contenido
                    for key in kpi.split("."):
                        val = val[key]
                    if isinstance(val, (int, float)):
                        data[kpi].append(val)
                except (KeyError, TypeError):
                    continue

    resumen = {"_info": {"n": n, "nivel_confianza": f"{nivel_confianza}%", "distribucion": ""}}

    if n > 100:
        z_val = norm.ppf((1 + prob) / 2)
        resumen["_info"]["distribucion"] = "normal"
    else:
        resumen["_info"]["distribucion"] = "t_student"

    for kpi, valores in data.items():
        arr = np.array(valores)
        mean = np.mean(arr)
        std = np.std(arr, ddof=1)
        sem = std / np.sqrt(n)
        if n > 100:
            error = z_val * sem
        else:
            t_val = t.ppf((1 + prob) / 2, df=n - 1)
            error = t_val * sem
        valor_str = f"{round(mean, 2)} ± {round(error, 2)}"
        if n == 1:
            valor_str = f"{mean}"

        puntero = resumen
        keys = kpi.split(".")
        for key in keys[:-1]:
            puntero = puntero.setdefault(key, {})
        puntero[keys[-1]] = valor_str

    if guardar_en:
        with open(guardar_en, "w") as f:
            json.dump(resumen, f, indent=4)

    return resumen

In [3]:
def parse_valor_ic(valor):
    """Convierte un string '123.4 ± 2.3' en una tupla (valor, error)."""
    match = re.match(r"([-\d.]+)\s*±\s*([\d.]+)", valor)
    if match:
        valor = float(match.group(1))
        error = float(match.group(2))
        return valor, error
    else:
        raise ValueError(f"No se pudo parsear el valor con IC: {valor}")

def parse_valor_base(valor):
    """Convierte un string como '123.4' en float."""
    try:
        return float(valor)
    except Exception:
        raise ValueError(f"No se pudo convertir el valor base: {valor}")

def diferencia_intervalo(base, valor_ic_con_error):
    """Calcula (base - (valor + error), base - (valor - error))."""
    val_ic, error = parse_valor_ic(valor_ic_con_error)
    diff_min = round(base - (val_ic + error), 2)
    diff_max = round(base - (val_ic - error), 2)
    return (diff_min, diff_max)

def restar_dicts_base_menos_ic(dict_base, dict_ic):
    """Resta dict_base - dict_ic en todos los niveles, dejando tuplas (min, max)."""
    resultado = {}
    for key in dict_ic:
        if isinstance(dict_ic[key], dict) and isinstance(dict_base.get(key), dict):
            resultado[key] = restar_dicts_base_menos_ic(dict_base[key], dict_ic[key])
        elif isinstance(dict_ic[key], str) and isinstance(dict_base.get(key), str):
            try:
                base = parse_valor_base(dict_base[key])
                resultado[key] = diferencia_intervalo(base, dict_ic[key])
            except ValueError:
                resultado[key] = None
        else:
            resultado[key] = None
    return resultado

In [4]:
def escape_latex(s):
    """
    Escapa los caracteres especiales de LaTeX en un string.
    """
    if not isinstance(s, str):
        s = str(s)
    replacements = {
        "\\": r"\\textbackslash{}",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "$": r"\$",
        "&": r"\&",
        "%": r"\%",
        "#": r"\#",
        "^": r"\^{}",
        "~": r"\~{}",
    }
    pattern = re.compile("|".join(re.escape(k) for k in replacements.keys()))
    return pattern.sub(lambda m: replacements[m.group()], s)

def dict_to_latex_string(d, indent=0):
    """
    Recursivamente convierte un dict anidado en un string con formato tipo código,
    escapando los caracteres peligrosos para LaTeX.
    """
    lines = []
    indent_str = "  " * indent
    for key, value in d.items():
        key_str = escape_latex(repr(key))
        if isinstance(value, dict):
            lines.append(f"{indent_str}{key_str}: "+"{")
            lines.extend(dict_to_latex_string(value, indent + 1))
            lines.append(f"{indent_str}"+"},")
        else:
            if isinstance(value, str):
                val_str = f"'{escape_latex(value)}'"
            else:
                val_str = escape_latex(repr(value))
            lines.append(f"{indent_str}{key_str}: {val_str},")
    return lines

def generar_latex_dict_texto(diccionario):
    """
    Devuelve un string listo para incluir en LaTeX.
    """
    return "\n".join(dict_to_latex_string(diccionario))

In [5]:
def save_nested_dict_tabbed_to_txt(d, filename="txt_kpis/kpis_output.txt"):
    os.makedirs("txt_kpis", exist_ok=True)

    lines = []

    def recurse(d, level=0):
        for i, (key, value) in enumerate(d.items()):
            indent = '\t' * level
            if isinstance(value, dict):
                lines.append(f"{indent}{key}:")
                recurse(value, level + 1)
                if level == 0:  # Add a blank line between top-level sections
                    lines.append("")
            else:
                lines.append(f"{indent}{key}: {value}")

    recurse(d)

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"✅ Dict saved to '{filename}' in tabbed format.")

In [6]:
resumen_proactivo = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloProactivo_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
resumen_base = calcular_kpis_promedio_con_ic_en_formato_json("resultados simulacion/ModeloBase_T4500_C4208/kpis", nivel_confianza=99, guardar_en=None)
diferencias = restar_dicts_base_menos_ic(resumen_base, resumen_proactivo)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
save_nested_dict_tabbed_to_txt(diferencias)

✅ Dict saved to 'txt_kpis/kpis_output.txt' in tabbed format.
